## Instalaciones de dependencias 

In [2]:
!pip install pandas scikit-learn tabulate imbalanced-learn


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Importación de librerías

In [3]:
# Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from imblearn.over_sampling import SMOTE
from tabulate import tabulate
print(" Librerías cargadas correctamente")

 Librerías cargadas correctamente


## Cargar dataset limpio

In [4]:
import pandas as pd

# Cargar directamente el CSV desde la carpeta local
df = pd.read_csv("telco_limpio.csv")

print(f"Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head(3)


Dataset cargado: 7043 filas x 20 columnas


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,0,1,0,-1.277445,0,1,0,0,2,0,0,0,0,0,1,2,-1.160323,-0.994242,0
1,1,0,0,0,0.066327,1,0,0,2,0,2,0,0,0,1,0,3,-0.259629,-0.173244,0
2,1,0,0,0,-1.236724,1,0,0,2,2,0,0,0,0,0,1,3,-0.362660,-0.959674,1


## Separar X e y

In [5]:
#Separar variables independientes (X) y variable objetivo (y)
X = df.drop(columns=['Churn'])
y = df['Churn']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nDistribución de Churn:")
print(y.value_counts())
print(f"\nDesbalanceo: {round(y.value_counts()[0]/y.value_counts()[1], 2)}:1")

X shape: (7043, 19)
y shape: (7043,)

Distribución de Churn:
Churn
0    5174
1    1869
Name: count, dtype: int64

Desbalanceo: 2.77:1


##  Balanceo de clases con **SMOTE**

In [6]:
# Balancear dataset con SMOTE (sobremuestreo de clase minoritaria)
smote = SMOTE(random_state=42)
X_bal, y_bal = smote.fit_resample(X, y)

print(f"Antes del SMOTE: {y.value_counts().to_dict()}")
print(f"Después del SMOTE: {pd.Series(y_bal).value_counts().to_dict()}")
print(f" Dataset balanceado")

Antes del SMOTE: {0: 5174, 1: 1869}
Después del SMOTE: {0: 5174, 1: 5174}
 Dataset balanceado


## División train/test

In [7]:
# Celda 6 — División train 80% / test 20%
X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape[0]} filas")
print(f"Test:  {X_test.shape[0]} filas")
print(" División aplicada")

Train: 8278 filas
Test:  2070 filas
 División aplicada


## Entrenar Regresión Logística

In [8]:
# Entrenar Regresión Logística
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
auc_lr = roc_auc_score(y_test, lr.predict_proba(X_test)[:,1])

print("=== REGRESIÓN LOGÍSTICA ===")
print(classification_report(y_test, y_pred_lr))
print(f"AUC-ROC: {round(auc_lr, 4)}")

=== REGRESIÓN LOGÍSTICA ===
              precision    recall  f1-score   support

           0       0.83      0.73      0.78      1021
           1       0.76      0.86      0.81      1049

    accuracy                           0.79      2070
   macro avg       0.80      0.79      0.79      2070
weighted avg       0.80      0.79      0.79      2070

AUC-ROC: 0.8771


## Entrenar Random Forest

In [9]:
# Entrenar Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
auc_rf = roc_auc_score(y_test, rf.predict_proba(X_test)[:,1])

print("=== RANDOM FOREST ===")
print(classification_report(y_test, y_pred_rf))
print(f"AUC-ROC: {round(auc_rf, 4)}")

=== RANDOM FOREST ===
              precision    recall  f1-score   support

           0       0.86      0.82      0.84      1021
           1       0.83      0.87      0.85      1049

    accuracy                           0.85      2070
   macro avg       0.85      0.85      0.85      2070
weighted avg       0.85      0.85      0.85      2070

AUC-ROC: 0.9225


## Validación cruzada

In [12]:
# Validación cruzada (cross-validation k=5)
cv_lr = cross_val_score(lr, X_bal, y_bal, cv=5, scoring='roc_auc')
cv_rf = cross_val_score(rf, X_bal, y_bal, cv=5, scoring='roc_auc')

print("=== VALIDACIÓN CRUZADA (AUC-ROC) ===")
tabla = [
    ["Regresión Logística", round(cv_lr.mean(), 4), round(cv_lr.std(), 4)],
    ["Random Forest",       round(cv_rf.mean(), 4), round(cv_rf.std(), 4)]
]
print(tabulate(tabla, headers=["Modelo", "AUC Media", "Desv. Estándar"], tablefmt="psql"))

=== VALIDACIÓN CRUZADA (AUC-ROC) ===
+---------------------+-------------+------------------+
| Modelo              |   AUC Media |   Desv. Estándar |
|---------------------+-------------+------------------|
| Regresión Logística |      0.8646 |           0.022  |
| Random Forest       |      0.9153 |           0.0277 |
+---------------------+-------------+------------------+
